<h1 align="center">
  <span style="background: linear-gradient(90deg, #6383ecff, #2e28ccff); 
               -webkit-background-clip: text; 
               -webkit-text-fill-color: transparent;">
    SALVADO Y CARGA DE MODELOS 
  </span>
</h1>


---

Una vez se acaba el script de entrenamiento se pierde entonces hay que guardarlo...

La clase Model tienen en sus param aprendibles como <i>state_dict</i> los optimizadores tb lo tienen 

Python ya tiene soluciones para ello y se usa una librería llamada Pickle que es un estándar para serializar objetos 

-----------------

Importación de librerias 

In [121]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter  
from torch.utils.data import random_split
from torch.utils.data import Dataset
from random import sample
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error, r2_score



In [122]:
class StandardScaler:

    def __init__(self, mean=None, std=None, epsilon=1e-7):
        """Standard Scaler.
        The class can be used to normalize PyTorch Tensors using native functions. The module does not expect the
        tensors to be of any specific shape; as long as the features are the last dimension in the tensor, the module
        will work fine.
        :param mean: The mean of the features. The property will be set after a call to fit.
        :param std: The standard deviation of the features. The property will be set after a call to fit.
        :param epsilon: Used to avoid a Division-By-Zero exception.
        """
        self.mean = mean
        self.std = std
        self.epsilon = epsilon
    def fit(self, values):
        dims = list(range(values.dim() - 1))
        self.mean = torch.mean(values, dim=dims)
        self.std = torch.std(values, dim=dims)

    def transform(self, values):
        return (values - self.mean) / (self.std + self.epsilon)

    def fit_transform(self, values):
        self.fit(values)
        return self.transform(values)

    def __repr__(self):
        return f"mean: {self.mean}, std:{self.std}, epsilon:{self.epsilon}"

In [123]:
class IrisDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        df_iris= pd.read_csv(csv_file, header=None)

        df_iris.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

        # Mapear las clases a valores numéricos
        mapeo = {
            'Iris-setosa': 0, 
            'Iris-versicolor': 1, 
            'Iris-virginica': 2
        }

        # Reemplazamos el texto de la columna por los números correspondientes 
        df_iris['class'] = df_iris['class'].map(mapeo)

        # Dividir el dataset 
        X = df_iris.drop('class', axis=1).values
        y = df_iris['class'].values

        # Convertir a tensores 

        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        
        # Escalar datos 

        scaler = StandardScaler()
        X_scaled = torch.tensor(scaler.fit_transform(X_tensor), dtype=torch.float32)

        # unimos todo

        self.data = torch.cat((X_scaled, y_tensor.unsqueeze(1)), 1)

        # Guardar el directorio raíz y la transformación (si se proporciona)

        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)
    

    def __getitem__(self, idx):
        # esto indica que idx numero normal
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        
        preds = self.data[idx, :-1] 
        
        spcs = self.data[idx, -1].long()

        # empaquetar y devolver 
        sample= (preds, spcs)
        if self.transform: 
            sample = self.transform(sample)
        return sample
    



In [124]:
dataset_iris = IrisDataset(csv_file="../../docs/iris/iris.data", root_dir="../../docs/iris/")
display(dataset_iris.data)

/tmp/ipykernel_53930/779325988.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_scaled = torch.tensor(scaler.fit_transform(X_tensor), dtype=torch.float32)


tensor([[-8.9767e-01,  1.0286e+00, -1.3368e+00, -1.3086e+00,  0.0000e+00],
        [-1.1392e+00, -1.2454e-01, -1.3368e+00, -1.3086e+00,  0.0000e+00],
        [-1.3807e+00,  3.3672e-01, -1.3935e+00, -1.3086e+00,  0.0000e+00],
        [-1.5015e+00,  1.0609e-01, -1.2801e+00, -1.3086e+00,  0.0000e+00],
        [-1.0184e+00,  1.2592e+00, -1.3368e+00, -1.3086e+00,  0.0000e+00],
        [-5.3538e-01,  1.9511e+00, -1.1668e+00, -1.0465e+00,  0.0000e+00],
        [-1.5015e+00,  7.9798e-01, -1.3368e+00, -1.1776e+00,  0.0000e+00],
        [-1.0184e+00,  7.9798e-01, -1.2801e+00, -1.3086e+00,  0.0000e+00],
        [-1.7430e+00, -3.5517e-01, -1.3368e+00, -1.3086e+00,  0.0000e+00],
        [-1.1392e+00,  1.0609e-01, -1.2801e+00, -1.4396e+00,  0.0000e+00],
        [-5.3538e-01,  1.4899e+00, -1.2801e+00, -1.3086e+00,  0.0000e+00],
        [-1.2600e+00,  7.9798e-01, -1.2234e+00, -1.3086e+00,  0.0000e+00],
        [-1.2600e+00, -1.2454e-01, -1.3368e+00, -1.4396e+00,  0.0000e+00],
        [-1.8638e+00, -1.

In [125]:
lonxitudeDataset = len(dataset_iris)

tamTrain = int(lonxitudeDataset * 0.8)
tamVal = lonxitudeDataset - tamTrain  # así evitamos errores de redondeo

print(f"Tam dataset: {lonxitudeDataset} train: {tamTrain} val: {tamVal}")

train_set, val_set = random_split(dataset_iris, [tamTrain, tamVal])

train_ldr = torch.utils.data.DataLoader(train_set, batch_size=2, shuffle=True, drop_last=False)

validation_loader = torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)


Tam dataset: 150 train: 120 val: 30


In [126]:
class Model(nn.Module):
    def __init__(self, entradas):
        super(Model, self).__init__()
        self.layer1 = nn.Linear(entradas, 100)
        self.layer2 = nn.Linear(100, 50)
        self.layer3 = nn.Linear(in_features=50, out_features=3) 
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [127]:
# Iris dataset has 4 input features (sepal/petal length/width)
model     = Model(4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
display(model)

Model(
  (layer1): Linear(in_features=4, out_features=100, bias=True)
  (layer2): Linear(in_features=100, out_features=50, bias=True)
  (layer3): Linear(in_features=50, out_features=3, bias=True)
)

In [128]:
entradaProba,dest = next(iter(train_ldr))
print("Entrada:")
display(entradaProba)
print("Desexada:")
display(dest)
saida = model(entradaProba) # esta é a proba de verdade
print("Saída:")
display(saida)
loss_fn(saida, dest)


Entrada:


tensor([[-0.8977,  1.7205, -1.2234, -1.3086],
        [ 0.5515,  0.5674,  0.5335,  0.5259]])

Desexada:


tensor([0, 1])

Saída:


tensor([[-0.0534,  0.0746,  0.0789],
        [-0.0441,  0.0032, -0.0429]], grad_fn=<AddmmBackward0>)

tensor(1.1275, grad_fn=<NllLossBackward0>)

In [129]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.
    # usamos enumerate para saber en que batch imos
    for i, data in enumerate(train_ldr):
        # Every data instance is an input + label pair
        inputs, labels = data
        # Zero your gradients for every batch!
        optimizer.zero_grad()
        # Make predictions for this batch
        outputs = model(inputs)
        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        # Adjust learning weights
        optimizer.step()
        # Gather data and report
        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 10 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.
    return last_loss

In [130]:
EPOCHS = 100
loss_list     = torch.zeros((EPOCHS,))
accuracy_list = torch.zeros((EPOCHS,))

writer = SummaryWriter()

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch + 1))

    # Poñemos o modelo en modo entrenamento
    model.train(True)
    avg_loss = train_one_epoch(epoch, writer)
    loss_list[epoch] = avg_loss
    # Non se precisan os gradientes para o test
    model.train(False)

EPOCH 1:
  batch 10 loss: 0.9560214936733246
  batch 20 loss: 0.38644470926374197
  batch 30 loss: 0.20244788239142508
  batch 40 loss: 0.4151798712904565
  batch 50 loss: 0.3166015163064003
  batch 60 loss: 0.18475440577603877
EPOCH 2:
  batch 10 loss: 0.21103853583335877
  batch 20 loss: 0.410249633110287
  batch 30 loss: 0.14210680465912376
  batch 40 loss: 0.09096844519954139
  batch 50 loss: 0.2212785703316058
  batch 60 loss: 0.32969853240210795
EPOCH 3:
  batch 10 loss: 0.03947753048560117
  batch 20 loss: 0.20130795700861198
  batch 30 loss: 0.09791051008069189
  batch 40 loss: 0.037582597804521355
  batch 50 loss: 0.15042966401670127
  batch 60 loss: 0.14935477981343864
EPOCH 4:
  batch 10 loss: 0.03913696244997027
  batch 20 loss: 0.04433145081093244
  batch 30 loss: 0.010591036807454657
  batch 40 loss: 0.07063647834584116
  batch 50 loss: 0.11847559785837802
  batch 60 loss: 0.7094252167880754
EPOCH 5:
  batch 10 loss: 0.03224567667930387
  batch 20 loss: 0.1460374294780194

In [131]:
import torch
import pickle

torch.save(model.state_dict(), "modelo_iris.pth")


print("modelo_iris.pth guardado")


df_iris = pd.read_csv("../../docs/iris/iris.data", header=None)
X_tensor = torch.tensor(df_iris.iloc[:, :-1].values, dtype=torch.float32)
scaler = StandardScaler()
scaler.fit(X_tensor)

with open('scaler_iris.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("scaler_iris.pkl guardado")

modelo_iris.pth guardado
scaler_iris.pkl guardado


In [132]:

df_completo = pd.read_csv("../../docs/iris/iris.data", header=None)

df_proba = df_completo.iloc[:5, :-1]

df_proba.to_csv("iris_5filas.csv", index=False, header=False)

print("iris_5filas.csv guardado")

iris_5filas.csv guardado


-------

PRUEBA DE QUE FUNCIONE BIEN EL GUARDADO 

In [133]:

class Model(nn.Module):
    def __init__(self, entradas):
        super(Model, self).__init__()
        self.layer1 = nn.Linear(entradas, 100)
        self.layer2 = nn.Linear(100, 50)
        self.layer3 = nn.Linear(50, 3) 
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)


modelo_cargado = Model(4)
modelo_cargado.load_state_dict(torch.load("modelo_iris.pth", weights_only=True))
modelo_cargado.eval() 

with open('scaler_iris.pkl', 'rb') as f:
    scaler_cargado = pickle.load(f)



print("Leyendo el archivo de las 5 flores...")
df_nuevos = pd.read_csv("iris_5filas.csv", header=None)

raw_data = torch.tensor(df_nuevos.values, dtype=torch.float32)


scaled_data = torch.tensor(scaler_cargado.transform(raw_data), dtype=torch.float32)



inverse_map = {0: 'Iris-setosa', 1: 'Iris-versicolor', 2: 'Iris-virginica'}

print("\n--- RESULTADOS ---")

with torch.no_grad():

    predictions = modelo_cargado(scaled_data)
    
    _, final_predictions = torch.max(predictions, 1)

for i, winner in enumerate(final_predictions.tolist()):
    flower_name = inverse_map[winner]
    size = df_nuevos.iloc[i].values
    print(f"Flor {i+1} (Medidas: {size}) ---> dice que es una: {flower_name}")



Leyendo el archivo de las 5 flores...

--- RESULTADOS ---
Flor 1 (Medidas: [5.1 3.5 1.4 0.2]) ---> dice que es una: Iris-setosa
Flor 2 (Medidas: [4.9 3.  1.4 0.2]) ---> dice que es una: Iris-setosa
Flor 3 (Medidas: [4.7 3.2 1.3 0.2]) ---> dice que es una: Iris-setosa
Flor 4 (Medidas: [4.6 3.1 1.5 0.2]) ---> dice que es una: Iris-setosa
Flor 5 (Medidas: [5.  3.6 1.4 0.2]) ---> dice que es una: Iris-setosa


/tmp/ipykernel_53930/1322876663.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scaled_data = torch.tensor(scaler_cargado.transform(raw_data), dtype=torch.float32)
